# VIDEO STUDIO AI — Wan 2.2 TI2V-5B (GPU gratuita Kaggle)

Este cuaderno ejecuta **ComfyUI localmente dentro de tu sesión Kaggle** y genera un clip neuronal real con Wan 2.2. No usa Comfy Cloud, API de pago, túneles públicos ni tu computadora. La GPU gratuita está sujeta a disponibilidad y cuota de Kaggle.

**Primera prueba controlada:** 640×352, 49 fotogramas (~2 segundos a 24 fps), 12 pasos. Después se podrán subir resolución, pasos y duración si la GPU y el tiempo lo permiten.

**Importante:** el video se descarga como archivo de salida del cuaderno. Después, en VIDEO STUDIO AI, usa «Selecciona una imagen o video» y «Guardar imagen en la nube» para subir el MP4 al proyecto correcto. No pegues claves de Supabase en este cuaderno.

In [ ]:
import os, shutil, subprocess, sys, time, json, uuid, requests, torch
from pathlib import Path

print('Python:', sys.version.split()[0])
print('CUDA disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle no asignó una GPU CUDA. En Settings > Accelerator selecciona GPU y vuelve a ejecutar.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM: {p.total_memory/1024**3:.1f} GiB')
print('Disco libre en /kaggle/working:', round(shutil.disk_usage('/kaggle/working').free/1024**3, 1), 'GiB')
print('Se utilizará una sola GPU. La VRAM de dos T4 no se suma automáticamente.')

## 1. Instalar ComfyUI y FFmpeg
No actualices PyTorch en Kaggle: se conserva la versión CUDA que proporciona el entorno.

In [ ]:
COMFY = Path('/kaggle/working/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY)], check=True)
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(COMFY/'requirements.txt')], check=True)
print('ComfyUI listo:', COMFY)

## 2. Descargar los tres modelos oficiales
La descarga requiere Internet habilitado en la configuración del cuaderno. Los archivos son grandes; comprueba el espacio disponible antes de iniciar.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-U','huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download
repo = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
model_specs = [
    ('split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors', COMFY/'models/diffusion_models'),
    ('split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', COMFY/'models/text_encoders'),
    ('split_files/vae/wan2.2_vae.safetensors', COMFY/'models/vae'),
]
for filename, folder in model_specs:
    folder.mkdir(parents=True, exist_ok=True)
    target = folder/Path(filename).name
    if target.exists() and target.stat().st_size > 100_000_000:
        print('Ya existe:', target.name)
        continue
    print('Descargando:', filename)
    downloaded = hf_hub_download(repo_id=repo, filename=filename, local_dir='/kaggle/working/wan22_download')
    shutil.copy2(downloaded, target)
    print('Listo:', target.name, round(target.stat().st_size/1024**3,2), 'GiB')
print('Modelos preparados. Espacio libre:', round(shutil.disk_usage('/kaggle/working').free/1024**3,1), 'GiB')

## 3. Iniciar ComfyUI en loopback
El servidor sólo escucha en `127.0.0.1`. No se crea un túnel ni se publica un endpoint en Internet.

In [ ]:
LOG = Path('/kaggle/working/comfyui.log')
log_handle = open(LOG,'w')
comfy_process = subprocess.Popen(
    [sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-auto-launch'],
    cwd=COMFY, stdout=log_handle, stderr=subprocess.STDOUT
)
BASE = 'http://127.0.0.1:8188'
for attempt in range(180):
    if comfy_process.poll() is not None:
        print(LOG.read_text(errors='ignore')[-6000:])
        raise RuntimeError('ComfyUI terminó durante el arranque.')
    try:
        if requests.get(BASE+'/system_stats', timeout=3).ok:
            print('ComfyUI está listo en loopback.')
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print(LOG.read_text(errors='ignore')[-6000:])
    raise TimeoutError('ComfyUI no respondió en 15 minutos.')

## 4. Crear un clip neuronal de prueba
Escribe una descripción visual concreta y describe el movimiento del sujeto y de la cámara. La primera prueba usa texto-a-video. Wan 2.2 también admite imagen-a-video; ese flujo se incorporará después de validar el primer render.

In [ ]:
default_prompt = ('A realistic red sports car drives fast along a winding mountain road at sunset, '
                  'wheels spinning, suspension reacting to the road, trees and guardrails moving past, '
                  'dynamic tracking camera following the car, cinematic natural motion, realistic reflections.')
prompt_text = input('Describe la escena y su movimiento (Enter para usar el ejemplo): ').strip() or default_prompt
negative_text = ('low quality, blurry, static image, frozen motion, flicker, deformed objects, '
                 'distorted wheels, extra wheels, text, subtitles, watermark, oversaturated')
seed = int.from_bytes(os.urandom(4),'big')
api_workflow = {
 '3': {'class_type':'KSampler','inputs':{'cfg':5,'denoise':1,'latent_image':['55',0],'model':['48',0],'negative':['7',0],'positive':['6',0],'sampler_name':'uni_pc','scheduler':'simple','seed':seed,'steps':12}},
 '6': {'class_type':'CLIPTextEncode','inputs':{'clip':['38',0],'text':prompt_text}},
 '7': {'class_type':'CLIPTextEncode','inputs':{'clip':['38',0],'text':negative_text}},
 '8': {'class_type':'VAEDecode','inputs':{'samples':['3',0],'vae':['39',0]}},
 '37': {'class_type':'UNETLoader','inputs':{'unet_name':'wan2.2_ti2v_5B_fp16.safetensors','weight_dtype':'default'}},
 '38': {'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','device':'default','type':'wan'}},
 '39': {'class_type':'VAELoader','inputs':{'vae_name':'wan2.2_vae.safetensors'}},
 '48': {'class_type':'ModelSamplingSD3','inputs':{'model':['37',0],'shift':8}},
 '55': {'class_type':'Wan22ImageToVideoLatent','inputs':{'batch_size':1,'height':352,'length':49,'vae':['39',0],'width':640}},
 '57': {'class_type':'CreateVideo','inputs':{'bit_depth':'auto','codec':'none','color_space':'sRGB','fps':24,'images':['8',0]}},
 '58': {'class_type':'SaveVideo','inputs':{'filename_prefix':'video/VSA_Wan22_test','format':'auto','format.codec':'auto','video':['57',0]}}
}
response = requests.post(BASE+'/prompt', json={'prompt':api_workflow,'client_id':str(uuid.uuid4())}, timeout=120)
if not response.ok:
    raise RuntimeError('ComfyUI rechazó el workflow: '+response.text[:5000])
prompt_id = response.json()['prompt_id']
print('Trabajo enviado. ID:', prompt_id, '| semilla:', seed)
print('No cierres la sesión mientras se procesa. La inferencia puede tardar bastante en GPU compartida.')

## 5. Esperar el resultado y guardar el MP4

In [ ]:
deadline = time.time() + 3*60*60
history_item = None
while time.time() < deadline:
    if comfy_process.poll() is not None:
        raise RuntimeError('ComfyUI se cerró. Revisa /kaggle/working/comfyui.log')
    hist = requests.get(BASE+'/history/'+prompt_id, timeout=30).json()
    history_item = hist.get(prompt_id)
    if history_item:
        status = history_item.get('status',{})
        if status.get('completed'):
            break
        if status.get('status_str') == 'error':
            raise RuntimeError('Falló la generación. Detalles: '+json.dumps(status,ensure_ascii=False)[:5000])
    time.sleep(10)
else:
    raise TimeoutError('La generación excedió el tiempo de espera de este cuaderno.')
outputs = history_item.get('outputs',{})
video_items = outputs.get('58',{}).get('videos',[])
if not video_items:
    raise RuntimeError('El workflow terminó pero no reportó un video en el nodo SaveVideo. Salida: '+json.dumps(outputs)[:5000])
item = video_items[0]
download = requests.get(BASE+'/view', params={'filename':item['filename'],'subfolder':item.get('subfolder',''),'type':item.get('type','output')}, timeout=300)
download.raise_for_status()
OUT = Path('/kaggle/working/VIDEO_STUDIO_AI_Wan22_test.mp4')
OUT.write_bytes(download.content)
metadata = {'model':'Wan 2.2 TI2V-5B','workflow':'ComfyUI official API graph','prompt':prompt_text,'seed':seed,'width':640,'height':352,'frames':49,'fps':24,'steps':12,'file':str(OUT),'bytes':OUT.stat().st_size}
Path('/kaggle/working/VIDEO_STUDIO_AI_Wan22_test.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
print('CLIP GENERADO:', OUT, '| tamaño:', round(OUT.stat().st_size/1024**2,2), 'MiB')
print(json.dumps(metadata,ensure_ascii=False,indent=2))

## 6. Incorporar el video a VIDEO STUDIO AI
1. En Kaggle, abre el panel **Output** del cuaderno y descarga `VIDEO_STUDIO_AI_Wan22_test.mp4`.
2. Abre tu aplicación privada VIDEO STUDIO AI e inicia sesión.
3. En **Sube una imagen para tu proyecto**, selecciona el MP4 y el proyecto de destino.
4. Pulsa **Guardar imagen en la nube**. Aunque el texto del botón sea genérico, acepta videos y los registra como `asset_type=video`.
5. Comprueba que aparezca en **Imágenes y videos guardados** y que se reproduzca. Después podrá asignarse a un segmento desde el espacio de producción.

Este paso manual evita copiar claves privadas a Kaggle y utiliza las políticas RLS y la sesión normal de tu aplicación. El audio se añade en la etapa de montaje de VIDEO STUDIO AI; este workflow de Wan genera video sin audio.

In [ ]:
print('Archivo disponible para descargar:', OUT if 'OUT' in globals() and OUT.exists() else 'Aún no generado')
print('Registro de ComfyUI: /kaggle/working/comfyui.log')
# Al terminar y descargar el MP4, puedes detener el proceso:
if 'comfy_process' in globals() and comfy_process.poll() is None:
    comfy_process.terminate()
    print('ComfyUI detenido.')